In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion

In [2]:
device = 'cpu'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "model_checkpoint.pth"
custom_model, custom_processor = load_trained_model(custom_model_name)

Loaded trained model from checkpoint.


In [4]:
import pandas as pd

songs = pd.read_pickle("./data/Songs")

In [33]:
vc = VocalAssistant(1)
vc.talk("What is your mood today?")
while True:
    command, vocal_file = vc.take_command()
    print(command)
    break

print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0]
print(audeering)
print()
print("Custom: ")
custom = list(predict_emotion(custom_model, custom_processor, vocal_file).values())
print(custom)

  listening....
Sample rate: 16000
Numpy array shape: (51270,)


Python(33935) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


i am super happy
Audeering: 
[0.6948153  0.72840106 0.529718  ]

Custom: 
[0.5761123895645142, 0.630628228187561, 0.6509610414505005]


In [34]:
import numpy as np
dim_vec = np.array(custom[0:2])
songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

songs_list = songs_list.sort_values(by="eucl_dist")[:5]
songs_list

#Careful that in the dataset there are some duplicates
#In a Deam Metadata file there titles with \t

,id,eucl_dist,Valence,Arousal,title,artist,mp3_file
500,1649,0.003177,0.577778,0.633333,\tI Remember You\t,Nature\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
625,1813,0.003177,0.577778,0.633333,\tWild & You\t,My Bubba & Mi\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
134,166,0.005737,0.575000,0.625000,Hypnotised,Coldplay,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
448,577,0.006961,0.575000,0.637500,Morning Light (The Voice Performance),Laith Al-Saadi,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
285,1336,0.012644,0.566667,0.622222,\tRock n Roll McDonald's\t,The Shut-Ins\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [32]:
import sounddevice as sd

for i in range(len(songs_list)):
    sd.play(songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()
